In [ ]:
import os
import sys
import subprocess
from pathlib import Path

# ==================================================================
# HARD-CODED POSIT CONNECT SETTINGS
# ==================================================================

CONNECT_URL = "https://ds-connect-prd.stc.com.sa/"
CONNECT_CONTENT_GUID = "2bdad5d0-f0d4-4832-a849-333f58b97a3d"
CONNECT_API_KEY = "q3wgD8XWKNFOWG7YMgA9J69mSY4UmawA"

# ==================================================================
# HARD-CODED APPLICATION ENVIRONMENT VARIABLES
# ==================================================================

APP_ENV = {
    # Select "glm" or "minmax"
    "LLM_PROVIDER": "glm",
    "MINIMAX_STREAM": "true",

    # Required only when LLM_PROVIDER="glm"
    "GLM_ENDPOINT": "https://litellm.apps.coherecls02.stc.corp/chat/completions",
    "GLM_MODEL": "GLM-5.2",
    "GLM_API_KEY": "sk-H2PyHg9YjjX4B5LuMUnIoQ",
    "GLM_STREAM": "true",
    "GLM_TIMEOUT_SECONDS": "1800",
    "GLM_MAX_RETRIES": "2",
    "GLM_RETRY_BACKOFF_SECONDS": "2",

    # Required only when LLM_PROVIDER="minmax"
    "MINIMAX_ENDPOINT": "https://minimax-m2-7-runai-aiservice.apps.rwbgpucls01.stc.corp/v1",
    "MINIMAX_MODEL": "MiniMaxAI/MiniMax-M2.7",
    "MINIMAX_API_KEY": "not-needed",

    # Teradata connection
    "TERADATA_USER": "AP_CVMBIGDATA",
    "TERADATA_PASSWORD": "G15c6n19$",
    "TERADATA_HOST_NAME": "172.21.237.135",

    # Speech-to-text service
    "PPQA_ASR_ENDPOINT": (
        "https://stc-asr-api-runai-aibuild.apps.rwbgpucls01.stc.corp"
    ),
    "PPQA_ASR_VERIFY_SSL": "false",
    "PPQA_ASR_TIMEOUT_SECONDS": "120",

    # Recommended for Posit Connect: use "teradata"
    "chat_db": "teradata",
    "CHAT_TERADATA_DATABASE": "DP_EDW_PPF_STG",

    # Only used when chat_db="local"
    "Host": "",
    "Port": "",
    "Database": "",
    "Username": "",
    "Password": "",
}


# ------------------------------------------------------------------
# Prepare the deployment environment
# ------------------------------------------------------------------

deployment_variables = {
    name: str(value).strip()
    for name, value in APP_ENV.items()
    if str(value).strip()
}

if APP_ENV["chat_db"].strip().lower() == "teradata":
    for name in ("Host", "Port", "Database", "Username", "Password"):
        deployment_variables.pop(name, None)

rsconnect = Path(sys.executable).with_name(
    "rsconnect.exe" if os.name == "nt" else "rsconnect"
)

if not rsconnect.exists():
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "rsconnect-python"],
        check=True,
    )

process_environment = os.environ.copy()
process_environment.update(deployment_variables)
process_environment["CONNECT_API_KEY"] = CONNECT_API_KEY.strip()

command = [
    str(rsconnect),
    "deploy",
    "streamlit",
    "--server",
    CONNECT_URL.strip(),
    "--app-id",
    CONNECT_CONTENT_GUID.strip(),
    "--title",
    "Performance Planning Q&A",
    "--entrypoint",
    "posit_app.py",
    "--exclude",
    "deploy.ipynb",
]

for variable_name in sorted(deployment_variables):
    command.extend(["--environment", variable_name])

command.append(".")

subprocess.run(command, check=True, env=process_environment)
